# 00 — Sanity Checks

Non-interacting reference calculations and density calibration for the 1D attractive Hubbard model with long-range hopping $t(r) = t/r^\alpha$.

**Sections**
1. Dispersion relation $\varepsilon(k)$ for various $\alpha$
2. Density of states $D(\varepsilon)$ for various $\alpha$
3. Hopping amplitude decay $t(r) = 1/r^\alpha$
4. Density calibration: find $\mu^*$ such that $n_\sigma = 0.4$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import sys, os

# Make analysis package importable from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
from analysis.extract import SimParams, scan_density_vs_mu

plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

## 1. Dispersion relation $\varepsilon(k)$

For the 1D chain with long-range hopping:
$$\varepsilon(k) = -2 \sum_{r=1}^{R_{\max}} \frac{t}{r^\alpha} \cos(kr)$$

In [ ]:
def dispersion(k, alpha, Rmax=200):
    """Non-interacting dispersion for 1D LR Hubbard chain."""
    eps = np.zeros_like(k, dtype=float)
    for r in range(1, Rmax + 1):
        eps -= 2.0 / r**alpha * np.cos(r * k)
    return eps

k = np.linspace(-np.pi, np.pi, 500)
alphas = [0.5, 0.8, 1.0, 1.5, 2.0, 10.0]  # 10.0 ≈ NN limit
labels = [f'α={a}' if a < 5 else 'NN limit' for a in alphas]

fig, ax = plt.subplots(figsize=(7, 4))
for a, lbl in zip(alphas, labels):
    ax.plot(k / np.pi, dispersion(k, a), label=lbl)
ax.set_xlabel(r'$k / \pi$')
ax.set_ylabel(r'$\varepsilon(k)$')
ax.set_title('Dispersion relation')
ax.legend(fontsize=9)
ax.set_xlim(-1, 1)
plt.tight_layout()
plt.savefig('../results/dispersion_LR_1D.pdf')
plt.show()

## 2. Density of states $D(\varepsilon)$

In [ ]:
def dos(alpha, Rmax=200, n_k=5000, n_bins=200):
    """DOS via histogram of ε(k) over the Brillouin zone."""
    k = np.linspace(-np.pi, np.pi, n_k, endpoint=False)
    eps = dispersion(k, alpha, Rmax)
    counts, edges = np.histogram(eps, bins=n_bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, counts

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=False)
for ax, a in zip(axes.flat, alphas):
    e, d = dos(a)
    ax.plot(e, d)
    ax.set_title(f'α = {a}')
    ax.set_xlabel(r'$\varepsilon$')
    ax.set_ylabel(r'$D(\varepsilon)$')
plt.suptitle('Density of states', y=1.01)
plt.tight_layout()
plt.savefig('../results/dos_LR_1D.pdf')
plt.show()

## 3. Hopping amplitude decay $t(r) = 1/r^\alpha$

In [ ]:
r = np.arange(1, 51)

fig, ax = plt.subplots(figsize=(6, 4))
for a in [0.5, 0.8, 1.0, 1.5, 2.0]:
    ax.plot(r, 1.0 / r**a, 'o-', ms=4, label=f'α={a}')
ax.set_xlabel(r'$r$')
ax.set_ylabel(r'$t(r) = 1/r^\alpha$')
ax.set_title('Hopping amplitude vs distance')
ax.legend()
ax.set_yscale('log')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('../results/hopping_amplitudes_LR_1D.pdf')
plt.show()

## 4. Density calibration — find $\mu^*$ for $n_\sigma = 0.4$

Run a μ sweep at the target parameters and interpolate to find the chemical
potential that gives $n_\sigma = 0.4$.

> **Before running**: make sure you have submitted and retrieved the density-scan
> simulations on the cluster. Set `DATA_ROOT` or pass `base=` below.

In [ ]:
# ---- Parameters for calibration scan ----
U    = -5.0
beta = 40.0
L    = 100
alpha = 2.0   # NN (Rmax=1) first
Rmax  = 1

# μ grid to scan (submit these with run_simulation.py --sweep-mu ...)
mu_range = np.arange(-3.5, -1.0, 0.25)

# Path to cluster data (adjust as needed)
DATA_BASE = "/nfs/home/gissa/Hubbard"

result = scan_density_vs_mu(
    U=U, beta=beta, L=L, mu_range=mu_range,
    alpha=alpha, Rmax=Rmax, sID=1, base=DATA_BASE
)

In [ ]:
# Interpolate to find mu* for n_sigma = 0.4
TARGET = 0.4  # n_sigma target

if len(result['mu']) > 1:
    # density per spin = total_density / 2
    n_sigma = result['density_up']  # or result['density'] / 2
    f_interp = interp1d(n_sigma, result['mu'], kind='linear', fill_value='extrapolate')
    mu_star = float(f_interp(TARGET))
    print(f"μ* for n_σ = {TARGET}: {mu_star:.4f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(result['mu'], n_sigma, 'o-')
    ax.axhline(TARGET, color='r', ls='--', label=f'n_σ = {TARGET}')
    ax.axvline(mu_star, color='g', ls='--', label=f'μ* = {mu_star:.3f}')
    ax.set_xlabel(r'$\mu$')
    ax.set_ylabel(r'$n_\sigma$')
    ax.set_title(f'Density calibration (U={U}, β={beta}, L={L}, α={alpha}, Rmax={Rmax})')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/density_calibration_NN.pdf')
    plt.show()
else:
    print("Not enough data points — run the μ sweep first.")